# Experiment 3 — TimeGAN Data Augmentation (Time Series)

**QUIDA multimodal fall-detection dataset.** Same pipeline as the VAE experiment, but the generator is
**TimeGAN** (Yoon et al., 2019). All four datasets (`X_imu`, `X_imu_ir`, `X_imu_radar`, `X_imu_ir_radar`)
are processed as **four completely independent experiments** — never merged.

TimeGAN has five networks — **embedder, recovery, generator, supervisor, discriminator** — trained in
three phases: (1) embedding autoencoder, (2) supervised next-step loss, (3) joint adversarial + supervised
+ moment-matching. Training uses explicit `GradientTape` loops (no `model.fit`), which is the natural way
to write TimeGAN and also avoids the Keras 3 `train_step` pitfalls.

Outputs go to `timegan_generated/` so they sit next to `vae_generated/` for a later method comparison.

## 1. Import libraries

In [ ]:
from __future__ import annotations

import os, json, pickle, random, warnings, time
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import rbf_kernel
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import pdist

import tensorflow as tf
try:
    import keras
    from keras import layers
except ImportError:
    from tensorflow import keras
    from tensorflow.keras import layers

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU") or "none (CPU)")


## 2. Configuration

`hidden_dim` is the TimeGAN latent size (independent of the number of features, so it copes with the very
wide datasets). Iteration counts are modest so the notebook completes on CPU — increase them
(2000-10000 each) for higher-quality synthetic data.

In [ ]:
@dataclass
class Config:
    # --- data ---
    data_dir: Path = field(default_factory=lambda: Path.cwd())
    output_dir: Path = field(default_factory=lambda: Path.cwd() / "timegan_generated")
    dataset_glob: str = "*_raw.npy"
    datasets: dict | None = None          # None -> auto-discover; or explicit {name: filename}

    # --- reproducibility / split ---
    seed: int = 42
    val_split: float = 0.2

    # --- TimeGAN model ---
    hidden_dim: int = 24                   # latent size (also noise dim); independent of n_features
    num_layers: int = 3
    gamma: float = 1.0                     # weight on the "e" adversarial term

    # --- training (iterations per phase) ---
    embed_iters: int = 400
    supervised_iters: int = 400
    joint_iters: int = 400
    batch_size: int = 128
    learning_rate: float = 1e-3

    # --- generation ---
    synthetic_ratio: float = 1.0           # n_synthetic = ratio * n_real

    # --- evaluation ---
    run_tsne: bool = True
    tsne_max_points: int = 500
    eval_subsample: int = 600

CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)
print("Data dir  :", CFG.data_dir)
print("Output dir:", CFG.output_dir)
print("Discovery :", CFG.dataset_glob if CFG.datasets is None else "explicit list")


In [ ]:
def set_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)

set_seeds(CFG.seed)


## 3. Automatically discover and load the four datasets

Each `*_raw.npy` in `data_dir` is loaded as its **own independent dataset** — never concatenated.

In [ ]:
def _clean_name(filename: str) -> str:
    stem = Path(filename).stem
    return stem[:-4] if stem.endswith("_raw") else stem

def discover_datasets(cfg: Config) -> dict[str, np.ndarray]:
    if cfg.datasets:
        items = list(cfg.datasets.items())
        missing = [str(cfg.data_dir / f) for _, f in items if not (cfg.data_dir / f).exists()]
        if missing:
            raise FileNotFoundError("Missing dataset file(s):\n  " + "\n  ".join(missing))
    else:
        files = sorted(cfg.data_dir.glob(cfg.dataset_glob),
                       key=lambda q: (q.name.count("_"), q.name))
        if not files:
            raise FileNotFoundError(f"No files matching {cfg.dataset_glob!r} in {cfg.data_dir}")
        items = [(_clean_name(q.name), q.name) for q in files]
    data = {}
    for name, fname in items:
        arr = np.load(cfg.data_dir / fname, allow_pickle=False).astype(np.float32)
        data[name] = arr
        print(f"  discovered {name:16s} <- {fname:26s} shape={arr.shape}")
    print(f"\n  {len(data)} independent dataset(s) - each trained separately, never merged.")
    return data

RAW = discover_datasets(CFG)


## 4. Data validation

In [ ]:
def load_metadata(cfg: Config, name: str):
    for cand in (f"{name}_meta.json", f"{name}.json", f"{name}_meta.csv"):
        q = cfg.data_dir / cand
        if q.exists():
            return json.loads(q.read_text()) if q.suffix == ".json" else pd.read_csv(q).to_dict("list")
    return None

def validate_dataset(name: str, X: np.ndarray, cfg: Config) -> np.ndarray:
    assert X.ndim == 3, f"{name}: expected 3-D (N, T, F), got {X.shape}"
    N, T, F = X.shape
    assert N > 0 and T > 0 and F > 0, f"{name}: empty axis in {X.shape}"
    n_nan, n_inf = int(np.isnan(X).sum()), int(np.isinf(X).sum())
    if n_nan or n_inf:
        print(f"  ! {name}: {n_nan} NaN / {n_inf} Inf -> imputing per-feature mean")
        X = np.where(np.isfinite(X), X, np.nan)
        feat_mean = np.nan_to_num(np.nanmean(X.reshape(-1, F), axis=0))
        inds = np.where(~np.isfinite(X))
        X[inds] = np.take(feat_mean, inds[2])
    meta = load_metadata(cfg, name)
    meta_msg = f"metadata={list(meta)}" if meta else "metadata=none"
    print(f"  {name:16s} N={N:<5d} T={T:<4d} F={F:<5d} "
          f"min={X.min():+.3f} max={X.max():+.3f} mean={X.mean():+.3f} | {meta_msg}")
    if F > N or N < 200:
        print(f"    [readiness] {name}: {N} samples for {F} features -> TimeGAN will strain / overfit; "
              f"treat synthetic data as approximate (more data / more iters help).")
    return X.astype(np.float32)

RAW = {name: validate_dataset(name, X, CFG) for name, X in RAW.items()}


## 5. Train / Validation split (+ per-dataset scaling)

Each dataset gets its own split and its own `MinMaxScaler` fit on **training data only**, then reshaped
back to `(N, T, F)`. Data is scaled to `[0, 1]` to match the sigmoid outputs of the networks.

In [ ]:
def split_and_scale(X: np.ndarray, cfg: Config):
    X_tr, X_val = train_test_split(X, test_size=cfg.val_split, random_state=cfg.seed, shuffle=True)
    N, T, F = X_tr.shape
    scaler = MinMaxScaler()
    scaler.fit(X_tr.reshape(-1, F))
    def apply(a):
        n = len(a)
        return scaler.transform(a.reshape(-1, F)).reshape(n, T, F).astype(np.float32)
    return apply(X_tr), apply(X_val), scaler, (T, F)

def inverse_scale(scaler, X_scaled):
    n, T, F = X_scaled.shape
    return scaler.inverse_transform(X_scaled.reshape(-1, F)).reshape(n, T, F)


## 6. Build TimeGAN

Five GRU-based networks. `embedder`/`recovery` form an autoencoder between data space `(T, F)` and latent
space `(T, hidden_dim)`; `generator` maps noise to latent codes; `supervisor` predicts the next latent
step; `discriminator` separates real vs generated latent sequences. Sigmoid outputs keep everything in
`[0, 1]`; the discriminator returns logits.

In [ ]:
def _gru_net(in_dim, out_dim, T, hidden, n_layers, out_act, name):
    x = keras.Input((T, in_dim))
    h = x
    for _ in range(n_layers):
        h = layers.GRU(hidden, return_sequences=True)(h)
    h = layers.Dense(out_dim, activation=out_act)(h)
    return keras.Model(x, h, name=name)

def build_discriminator(T, hidden, n_layers):
    x = keras.Input((T, hidden))
    h = x
    for _ in range(n_layers):
        h = layers.Bidirectional(layers.GRU(hidden, return_sequences=True))(h)
    h = layers.Dense(1)(h)                      # logits
    return keras.Model(x, h, name="discriminator")


class TimeGAN:
    """Container for the five networks + their optimizers + generation."""
    def __init__(self, T, F, cfg: Config):
        h, L = cfg.hidden_dim, cfg.num_layers
        self.T, self.F, self.hidden_dim, self.z_dim = T, F, h, h
        self.embedder     = _gru_net(F, h, T, h, L, "sigmoid", "embedder")
        self.recovery     = _gru_net(h, F, T, h, L, "sigmoid", "recovery")
        self.generator    = _gru_net(h, h, T, h, L, "sigmoid", "generator")   # noise dim = h
        self.supervisor   = _gru_net(h, h, T, h, max(1, L - 1), "sigmoid", "supervisor")
        self.discriminator = build_discriminator(T, h, L)
        lr = cfg.learning_rate
        self.opt_er = keras.optimizers.Adam(lr)   # embedder + recovery
        self.opt_s  = keras.optimizers.Adam(lr)   # supervisor (phase 2)
        self.opt_g  = keras.optimizers.Adam(lr)   # generator + supervisor (phase 3)
        self.opt_d  = keras.optimizers.Adam(lr)   # discriminator

    def generate(self, scaler, n):
        Z = np.random.uniform(0, 1, (n, self.T, self.z_dim)).astype("float32")
        E_hat = self.generator.predict(Z, batch_size=256, verbose=0)
        H_hat = self.supervisor.predict(E_hat, batch_size=256, verbose=0)
        X_hat = self.recovery.predict(H_hat, batch_size=256, verbose=0)   # [0, 1]
        return inverse_scale(scaler, X_hat).astype("float32"), X_hat

def build_timegan(T, F, cfg: Config) -> TimeGAN:
    return TimeGAN(T, F, cfg)


## 7. Train TimeGAN (three phases)

Explicit `GradientTape` steps (compiled with `@tf.function`). Phase 1 trains the autoencoder; phase 2 the
supervisor; phase 3 alternates generator, joint-embedder, and discriminator updates (the discriminator is
only stepped when its loss exceeds 0.15, the standard trick to stop it dominating). Validation
reconstruction loss is logged throughout.

In [ ]:
def train_timegan(gan: TimeGAN, X_tr, X_val, cfg: Config):
    bce = keras.losses.BinaryCrossentropy(from_logits=True)
    mse = keras.losses.MeanSquaredError()
    gamma = float(cfg.gamma)
    bs = min(cfg.batch_size, len(X_tr))
    Xv = tf.convert_to_tensor(X_val.astype("float32"))

    @tf.function
    def emb_step(X):
        with tf.GradientTape() as tape:
            H = gan.embedder(X); Xt = gan.recovery(H)
            e_t0 = mse(X, Xt); e0 = 10.0 * tf.sqrt(e_t0 + 1e-8)
        v = gan.embedder.trainable_variables + gan.recovery.trainable_variables
        gan.opt_er.apply_gradients(zip(tape.gradient(e0, v), v)); return e_t0

    @tf.function
    def sup_step(X):
        with tf.GradientTape() as tape:
            H = gan.embedder(X); Hs = gan.supervisor(H)
            ls = mse(H[:, 1:, :], Hs[:, :-1, :])
        v = gan.supervisor.trainable_variables
        gan.opt_s.apply_gradients(zip(tape.gradient(ls, v), v)); return ls

    @tf.function
    def gen_step(X, Z):
        with tf.GradientTape() as tape:
            H = gan.embedder(X); e_hat = gan.generator(Z); h_hat = gan.supervisor(e_hat)
            h_hat_s = gan.supervisor(H); x_hat = gan.recovery(h_hat)
            y_f = gan.discriminator(h_hat); y_fe = gan.discriminator(e_hat)
            l_u = bce(tf.ones_like(y_f), y_f); l_ue = bce(tf.ones_like(y_fe), y_fe)
            l_s = mse(H[:, 1:, :], h_hat_s[:, :-1, :])
            mx, vx = tf.nn.moments(X, axes=[0]); mxh, vxh = tf.nn.moments(x_hat, axes=[0])
            l_v = (tf.reduce_mean(tf.abs(tf.sqrt(vxh + 1e-6) - tf.sqrt(vx + 1e-6)))
                   + tf.reduce_mean(tf.abs(mxh - mx)))
            g = l_u + gamma * l_ue + 100.0 * tf.sqrt(l_s + 1e-8) + 100.0 * l_v
        v = gan.generator.trainable_variables + gan.supervisor.trainable_variables
        gan.opt_g.apply_gradients(zip(tape.gradient(g, v), v)); return g

    @tf.function
    def emb_joint_step(X):
        with tf.GradientTape() as tape:
            H = gan.embedder(X); Xt = gan.recovery(H)
            e_t0 = mse(X, Xt); e0 = 10.0 * tf.sqrt(e_t0 + 1e-8)
            Hs = gan.supervisor(H); l_s = mse(H[:, 1:, :], Hs[:, :-1, :])
            e = e0 + 0.1 * l_s
        v = gan.embedder.trainable_variables + gan.recovery.trainable_variables
        gan.opt_er.apply_gradients(zip(tape.gradient(e, v), v)); return e_t0

    @tf.function
    def disc_loss(X, Z):
        H = gan.embedder(X); e_hat = gan.generator(Z); h_hat = gan.supervisor(e_hat)
        y_r = gan.discriminator(H); y_f = gan.discriminator(h_hat); y_fe = gan.discriminator(e_hat)
        return (bce(tf.ones_like(y_r), y_r) + bce(tf.zeros_like(y_f), y_f)
                + gamma * bce(tf.zeros_like(y_fe), y_fe))

    @tf.function
    def disc_step(X, Z):
        with tf.GradientTape() as tape:
            d = disc_loss(X, Z)
        v = gan.discriminator.trainable_variables
        gan.opt_d.apply_gradients(zip(tape.gradient(d, v), v)); return d

    def batch():
        idx = np.random.choice(len(X_tr), bs, replace=len(X_tr) < bs)
        return tf.convert_to_tensor(X_tr[idx])
    def noise():
        return tf.convert_to_tensor(np.random.uniform(0, 1, (bs, gan.T, gan.z_dim)).astype("float32"))
    def val_recon():
        Xt = gan.recovery(gan.embedder(Xv, training=False), training=False)
        return float(mse(Xv, Xt))

    H = {"p1_step": [], "p1_emb": [], "p1_val": [], "p2_step": [], "p2_sup": [],
         "p3_step": [], "p3_G": [], "p3_D": [], "p3_emb": [], "p3_val": []}
    li = lambda n: max(1, n // 40)

    t0 = time.time()
    for it in range(1, cfg.embed_iters + 1):
        e = float(emb_step(batch()))
        if it % li(cfg.embed_iters) == 0 or it == cfg.embed_iters:
            H["p1_step"].append(it); H["p1_emb"].append(e); H["p1_val"].append(val_recon())
    print(f"    phase1 embedding  ({cfg.embed_iters} its) last recon={H['p1_emb'][-1]:.5f}")

    for it in range(1, cfg.supervised_iters + 1):
        s = float(sup_step(batch()))
        if it % li(cfg.supervised_iters) == 0 or it == cfg.supervised_iters:
            H["p2_step"].append(it); H["p2_sup"].append(s)
    print(f"    phase2 supervised ({cfg.supervised_iters} its) last sup={H['p2_sup'][-1]:.5f}")

    for it in range(1, cfg.joint_iters + 1):
        for _ in range(2):
            g = float(gen_step(batch(), noise()))
            e = float(emb_joint_step(batch()))
        Xb, Zb = batch(), noise()
        d = float(disc_loss(Xb, Zb))
        if d > 0.15:
            d = float(disc_step(Xb, Zb))
        if it % li(cfg.joint_iters) == 0 or it == cfg.joint_iters:
            H["p3_step"].append(it); H["p3_G"].append(g); H["p3_D"].append(d)
            H["p3_emb"].append(e); H["p3_val"].append(val_recon())
            print(f"      joint {it:4d}/{cfg.joint_iters} | G={g:7.3f} D={d:6.3f} recon={e:.4f}")
    print(f"    phase3 joint      ({cfg.joint_iters} its) done in {time.time()-t0:.1f}s")
    return H


## 8. Generate synthetic data

In [ ]:
# generation lives on the TimeGAN object (Z ~ U[0,1] -> generator -> supervisor -> recovery -> inverse scale)
def generate_samples(gan: TimeGAN, scaler, n, cfg: Config):
    return gan.generate(scaler, n)


## 9. Evaluate (real vs synthetic)

Same metric suite as the VAE experiment so results are directly comparable: **MMD** (multi-bandwidth RBF),
**Wasserstein**, per-feature **mean/std** error, **correlation preservation**, and a **diversity ratio**
(mode-collapse guard), plus the TimeGAN training losses. Visuals: loss curves, PCA, t-SNE, distributions.

In [ ]:
def _flat(a, cap, rng):
    a = a.reshape(len(a), -1)
    if len(a) > cap:
        a = a[rng.choice(len(a), cap, replace=False)]
    return a

def mmd_rbf(real, synth, cap, rng):
    X, Y = _flat(real, cap, rng), _flat(synth, cap, rng)
    med = np.median(pdist(X[:200])) if len(X) > 1 else 1.0
    med = med if med > 0 else 1.0
    return float(np.mean([
        rbf_kernel(X, X, g/med**2).mean() + rbf_kernel(Y, Y, g/med**2).mean()
        - 2*rbf_kernel(X, Y, g/med**2).mean() for g in (0.5, 1.0, 2.0)]))

def wasserstein_feat(real, synth):
    F = real.shape[-1]
    return float(np.mean([wasserstein_distance(real[..., f].ravel(), synth[..., f].ravel())
                          for f in range(F)]))

def moment_errors(real, synth):
    return (float(np.mean(np.abs(real.mean((0,1)) - synth.mean((0,1))))),
            float(np.mean(np.abs(real.std((0,1))  - synth.std((0,1))))))

def corr_preservation(real, synth):
    F = real.shape[-1]
    Cr = np.corrcoef(real.reshape(-1, F).T); Cs = np.corrcoef(synth.reshape(-1, F).T)
    return float(np.linalg.norm(np.nan_to_num(Cr) - np.nan_to_num(Cs)))

def diversity(a, cap, rng):
    a = _flat(a, cap, rng)
    return float(pdist(a).mean()) if len(a) > 1 else 0.0

def _last(lst):
    return float(lst[-1]) if lst else float("nan")

def evaluate(real, synth, history, cfg: Config):
    rng = np.random.default_rng(cfg.seed)
    md_, sd_ = moment_errors(real, synth)
    div_r, div_s = diversity(real, cfg.eval_subsample, rng), diversity(synth, cfg.eval_subsample, rng)
    return {
        "final_embedding_loss":     _last(history["p3_emb"]) if history["p3_emb"] else _last(history["p1_emb"]),
        "final_val_embedding_loss": _last(history["p3_val"]) if history["p3_val"] else _last(history["p1_val"]),
        "final_supervised_loss":    _last(history["p2_sup"]),
        "final_generator_loss":     _last(history["p3_G"]),
        "final_discriminator_loss": _last(history["p3_D"]),
        "mmd":              mmd_rbf(real, synth, cfg.eval_subsample, rng),
        "wasserstein":      wasserstein_feat(real, synth),
        "mean_abs_err":     md_,
        "std_abs_err":      sd_,
        "corr_frobenius":   corr_preservation(real, synth),
        "diversity_real":   div_r,
        "diversity_synth":  div_s,
        "diversity_ratio":  float(div_s / div_r) if div_r else 0.0,
    }


In [ ]:
def make_plots(name, real, synth, history, out_dir: Path, cfg: Config):
    rng = np.random.default_rng(cfg.seed)
    # 1) TimeGAN loss curves
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(history["p1_step"], history["p1_emb"], label="train")
    if history["p1_val"]:
        ax[0].plot(history["p1_step"], history["p1_val"], label="val")
    ax[0].set_title(f"{name} - embedding recon loss"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(history["p2_step"], history["p2_sup"])
    ax[1].set_title(f"{name} - supervised loss"); ax[1].set_xlabel("iter"); ax[1].grid(alpha=.3)
    ax[2].plot(history["p3_step"], history["p3_G"], label="generator")
    ax[2].plot(history["p3_step"], history["p3_D"], label="discriminator")
    ax[2].set_title(f"{name} - joint adversarial"); ax[2].set_xlabel("iter"); ax[2].legend(); ax[2].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(out_dir / "loss_curves.png", dpi=110); plt.show(); plt.close(fig)

    # 2) PCA + t-SNE (real vs synthetic)
    r = _flat(real, cfg.tsne_max_points, rng); s = _flat(synth, cfg.tsne_max_points, rng)
    X = np.vstack([r, s]); y = np.r_[np.zeros(len(r)), np.ones(len(s))]
    fig, ax = plt.subplots(1, 2 if cfg.run_tsne else 1, figsize=(12, 5), squeeze=False)
    p = PCA(2, random_state=cfg.seed).fit_transform(X)
    for lbl, m in [(0, "real"), (1, "synth")]:
        ax[0][0].scatter(*p[y==lbl].T, s=8, alpha=.5, label=m)
    ax[0][0].set_title(f"{name} - PCA"); ax[0][0].legend(); ax[0][0].grid(alpha=.3)
    if cfg.run_tsne:
        try:
            perp = max(5, min(30, len(X)//4))
            t = TSNE(2, perplexity=perp, init="pca", random_state=cfg.seed).fit_transform(X)
            for lbl, m in [(0, "real"), (1, "synth")]:
                ax[0][1].scatter(*t[y==lbl].T, s=8, alpha=.5, label=m)
            ax[0][1].set_title(f"{name} - t-SNE"); ax[0][1].legend(); ax[0][1].grid(alpha=.3)
        except Exception as e:
            ax[0][1].text(.5, .5, f"t-SNE skipped\n{e}", ha="center")
    fig.tight_layout(); fig.savefig(out_dir / "pca_tsne.png", dpi=110); plt.show(); plt.close(fig)

    # 3) per-feature distributions (up to 6)
    F = real.shape[-1]; k = min(F, 6)
    fig, ax = plt.subplots(1, k, figsize=(3.2*k, 3.2), squeeze=False)
    for f in range(k):
        ax[0][f].hist(real[..., f].ravel(), bins=40, alpha=.5, density=True, label="real")
        ax[0][f].hist(synth[..., f].ravel(), bins=40, alpha=.5, density=True, label="synth")
        ax[0][f].set_title(f"feat {f}"); ax[0][f].legend(fontsize=7)
    fig.suptitle(f"{name} - feature distributions"); fig.tight_layout()
    fig.savefig(out_dir / "distributions.png", dpi=110); plt.show(); plt.close(fig)


### Automated health checks (GAN-specific)

Flags **mode collapse** (low diversity), **discriminator dominance** (D-loss ≈ 0, so the generator gets no
signal), **generator instability** (diverging / NaN loss), and **overfitting** (val ≫ train reconstruction).

In [ ]:
def health_report(name, synth_scaled, metrics, cfg: Config):
    warns = []
    if synth_scaled.std() < 1e-3:
        warns.append("generator/recovery saturation (near-constant output)")
    if metrics["diversity_ratio"] < 0.5:
        warns.append("low diversity (possible mode collapse)")
    if metrics["final_discriminator_loss"] < 0.05:
        warns.append("discriminator dominating (D-loss ~ 0) - generator may not be learning")
    g = metrics["final_generator_loss"]
    if g != g or g > 1e3:
        warns.append("generator loss unstable / diverging")
    ve, te = metrics["final_val_embedding_loss"], metrics["final_embedding_loss"]
    if te and ve == ve and te == te and ve > 1.5 * te:
        warns.append("val >> train reconstruction (overfitting - expected with few samples)")
    if warns:
        for w in warns: print(f"    [warn] {name}: {w}")
    else:
        print(f"    [ok] {name}: no health issues flagged")
    return warns


## 10. Save generated datasets and evaluation outputs

Per dataset: `synthetic.npy`, `scaler.pkl`, `metadata.csv`, `metrics.json`, the five networks
(`generator/supervisor/recovery/embedder/discriminator.keras`), and the figures.

In [ ]:
def save_outputs(name, out_dir, synthetic, scaler, gan: TimeGAN, metrics, meta, warns):
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "synthetic.npy", synthetic)
    with open(out_dir / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    for nm, net in [("generator", gan.generator), ("supervisor", gan.supervisor),
                    ("recovery", gan.recovery), ("embedder", gan.embedder),
                    ("discriminator", gan.discriminator)]:
        try:
            net.save(out_dir / f"{nm}.keras")
        except Exception as e:
            print(f"    [note] could not save {nm}: {e}")
    pd.DataFrame([meta]).to_csv(out_dir / "metadata.csv", index=False)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump({**metrics, "warnings": warns}, f, indent=2)


## Run all four datasets (single execution)

Each dataset is an independent TimeGAN. A failure in one is caught and reported so the others still run.

In [ ]:
SEP = "=" * 50

def run_experiment(idx, total, name, X, cfg: Config):
    print(f"\n{SEP}\nEXPERIMENT {idx}/{total}: {name}  (independent TimeGAN)\n{SEP}")
    assert isinstance(X, np.ndarray) and X.ndim == 3, "each experiment trains on ONE dataset only"
    set_seeds(cfg.seed)
    out_dir = cfg.output_dir / name; out_dir.mkdir(parents=True, exist_ok=True)

    X_tr, X_val, scaler, (T, F) = split_and_scale(X, cfg)
    print(f"    split: train={len(X_tr)} val={len(X_val)} | T={T} F={F} | hidden_dim={cfg.hidden_dim}")

    gan = build_timegan(T, F, cfg)
    history = train_timegan(gan, X_tr, X_val, cfg)

    n_synth = int(round(cfg.synthetic_ratio * len(X)))
    synthetic, synth_scaled = generate_samples(gan, scaler, n_synth, cfg)
    print(f"    generated {len(synthetic)} synthetic samples, shape={synthetic.shape}")

    metrics = evaluate(X, synthetic, history, cfg)
    warns = health_report(name, synth_scaled, metrics, cfg)
    make_plots(name, X, synthetic, history, out_dir, cfg)

    meta = {"dataset": name, "n_real": len(X), "n_synth": len(synthetic), "seq_len": T,
            "n_features": F, "hidden_dim": cfg.hidden_dim, "num_layers": cfg.num_layers, **metrics}
    save_outputs(name, out_dir, synthetic, scaler, gan, metrics, meta, warns)
    print(f"    saved -> {out_dir}")
    return meta

def run_all(cfg: Config):
    rows, failures = [], {}
    items = list(RAW.items())
    for i, (name, X) in enumerate(items, 1):
        try:
            rows.append(run_experiment(i, len(items), name, X, cfg))
        except Exception as e:
            failures[name] = repr(e)
            print(f"    [ERROR] {name} failed: {e!r}")
    summary = pd.DataFrame(rows).set_index("dataset") if rows else pd.DataFrame()
    if not summary.empty:
        summary.to_csv(cfg.output_dir / "summary.csv")
    if failures:
        print("\nFailures:", failures)
    return summary

SUMMARY = run_all(CFG)
print("\n" + "=" * 50 + "\nCROSS-DATASET SUMMARY (independent TimeGANs)\n" + "=" * 50)
cols = ["n_real","n_synth","final_generator_loss","final_discriminator_loss","mmd","wasserstein",
        "mean_abs_err","std_abs_err","corr_frobenius","diversity_ratio"]
display(SUMMARY[[c for c in cols if c in SUMMARY.columns]] if not SUMMARY.empty else SUMMARY)


## Download the four synthetic datasets

Zips `timegan_generated/` (all `synthetic.npy` files, the five saved networks, scalers, metrics, charts)
into one archive.

In [ ]:
import shutil
if CFG.output_dir.exists():
    archive = shutil.make_archive(str(CFG.output_dir), "zip", root_dir=str(CFG.output_dir))
    print("Zipped ->", archive)
    print("\nSynthetic datasets produced:")
    for sub in sorted(CFG.output_dir.iterdir()):
        syn = sub / "synthetic.npy"
        if syn.exists():
            print(f"  {sub.name:16s} {syn}  shape={np.load(syn).shape}")


## Extending / comparing methods

`timegan_generated/` mirrors `vae_generated/`, so a later comparison notebook can load
`<method>_generated/<dataset>/synthetic.npy` and evaluate them side by side with the same metrics
(`mmd`, `wasserstein`, `mean_abs_err`, `std_abs_err`, `corr_frobenius`, `diversity_ratio`).

Quality knobs if the synthetic data looks poor: raise `embed_iters`/`supervised_iters`/`joint_iters`
(2000-10000 each), adjust `hidden_dim`, or tune `gamma`. Expect long CPU runtimes on the wide
(radar) datasets.